<a href="https://colab.research.google.com/github/DeepakSaini01/FlyRank-AI-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!git clone https://github.com/DeepakSaini01/FlyRank-AI-internship.git

Cloning into 'FlyRank-AI-internship'...
remote: Enumerating objects: 202, done.
remote: Counting objects: 100% (202/202), done.
remote: Compressing objects: 100% (152/152), done.
remote: Total 202 (delta 99), reused 100 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (202/202), 1.89 MiB | 5.42 MiB/s, done.
Resolving deltas: 100% (99/99), done.


In [4]:
import os, warnings
import pandas as pd, numpy as np
import pyarrow.dataset as ds
from huggingface_hub import login, HfApi, HfFileSystem
import matplotlib.pyplot as plt, seaborn as sns

warnings.filterwarnings("ignore")
sns.set(style="whitegrid")

# ------------------------------------------------------------
# 0️⃣ Auth
# ------------------------------------------------------------
HF_TOKEN = os.getenv("HF_TOKEN")

login(token=HF_TOKEN)

REPO = "FlyRank/internship-warehouse"
FOLDER = "fact_content_daily_performance"
TARGET_MONTH = "2026-03"

# ------------------------------------------------------------
# 1️⃣ List files, prune by the Hive partition key (month=...)
# ------------------------------------------------------------
api = HfApi(token=HF_TOKEN)
all_files = api.list_repo_files(REPO, repo_type="dataset")
daily_files = [f for f in all_files if f.startswith(f"{FOLDER}/") and f.endswith(".parquet")]
print(f"Found {len(daily_files)} parquet files in {FOLDER}/")

narrowed = [f for f in daily_files if f"month={TARGET_MONTH}" in f]
files_to_read = narrowed if narrowed else daily_files
print(f"Reading {len(files_to_read)} file(s): {files_to_read}")

if not files_to_read:
    raise FileNotFoundError(f"No files matched month={TARGET_MONTH} — check partition naming")

# ------------------------------------------------------------
# 2️⃣ Use HfFileSystem so PyArrow can actually open hf:// paths
# ------------------------------------------------------------
fs = HfFileSystem(token=HF_TOKEN)

# HfFileSystem paths are like "datasets/REPO/path/to/file.parquet" (no "hf://" prefix)
fs_paths = [f"datasets/{REPO}/{f}" for f in files_to_read]

cols = ["content_hash_id", "client_hash_id", "report_date",
        "gsc_impressions", "gsc_clicks", "gsc_avg_position"]

dataset = ds.dataset(fs_paths, filesystem=fs, format="parquet")
table = dataset.to_table(columns=cols)
df = table.to_pandas()

# ------------------------------------------------------------
# 3️⃣ Filter to target month (belt-and-suspenders, in case the
#    partition file itself contains a couple of stray dates)
# ------------------------------------------------------------
df = df[df["report_date"].astype(str).str.startswith(TARGET_MONTH)].copy()

# ------------------------------------------------------------
# 4️⃣ Rename columns
# ------------------------------------------------------------
df = df.rename(columns={
    "content_hash_id": "content_id",
    "client_hash_id":  "client_id",
    "gsc_impressions": "impressions_90d",
    "gsc_clicks":      "clicks_90d",
    "gsc_avg_position":"avg_position",
})

# ------------------------------------------------------------
# 5️⃣ Derive month, drop raw date
# ------------------------------------------------------------
df["month"] = pd.to_datetime(df["report_date"]).dt.strftime("%Y-%m")
df.drop(columns=["report_date"], inplace=True)

# ------------------------------------------------------------
# 6️⃣ Compute CTR
# ------------------------------------------------------------
df["ctr"] = np.where(
    df["impressions_90d"] > 0,
    df["clicks_90d"] / df["impressions_90d"],
    np.nan,
)
df["ctr"] = df["ctr"].fillna(df["ctr"].median())

# ------------------------------------------------------------
# 7️⃣ Sanity check
# ------------------------------------------------------------
print("\n✅ Daily slice ready")
print(" • Unique month(s) :", df["month"].unique())
print(" • Row count       :", df.shape[0])
print(" • Columns         :", df.columns.tolist())

Found 18 parquet files in fact_content_daily_performance/
Reading 1 file(s): ['fact_content_daily_performance/month=2026-03/data_0.parquet']

✅ Daily slice ready
 • Unique month(s) : ['2026-03']
 • Row count       : 9841378
 • Columns         : ['content_id', 'client_id', 'impressions_90d', 'clicks_90d', 'avg_position', 'month', 'ctr']


# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DeepakSaini01/ML-WEEK-1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row = one content page

Table used = flyrank_monthly

Time window = March 2026 (2026‑03)

Target = binary declining label (trend_direction == "down")

Exclude = internal ML‑system signals (provider_used, model_used, ai_sessions_90d)

In [5]:
# ------------------------------------------------------------
# 1️⃣ Verify that “one row = one content page” for March 2026
# ------------------------------------------------------------

# Keep only the development month (mid‑panel month)
march_mask = df["month"] == "2026-03"
march_df   = df.loc[march_mask].copy()

print("\n📅 Selected month:", march_df["month"].unique())
print("🧮 Number of rows for March 2026:", march_df.shape[0])

# Verify that the grain is (content_id, month)
grain_check = (
    march_df.groupby(["content_id", "month"])
    .size()
    .reset_index(name="rows_per_key")
)

print("\n🔎 Distinct (content_id, month) pairs:", grain_check.shape[0])
print("🔎 Any duplicate rows per key?", (grain_check["rows_per_key"] > 1).any())

# Show a few example rows so we can see the schema
display(march_df.head())


📅 Selected month: ['2026-03']
🧮 Number of rows for March 2026: 9841378

🔎 Distinct (content_id, month) pairs: 331437
🔎 Any duplicate rows per key? True


,content_id,client_id,impressions_90d,clicks_90d,avg_position,month,ctr
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,20,0,3.350000,2026-03,0.000
1,content_05597932fe4da067,client_73cda7b4e4f265ea,1,0,0.000000,2026-03,0.000
2,content_7a105f548d9c6916,client_73cda7b4e4f265ea,125,1,4.928000,2026-03,0.008
3,content_905aa32a0230694e,client_73cda7b4e4f265ea,7,0,4.000000,2026-03,0.000
4,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,11,0,2.272727,2026-03,0.000


## 2. Fields: feature / label / context / excluded

| Bucket      | Columns (why they belong)                                                                                                                                 |
|------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------|
| **Feature**| `impressions_90d`, `clicks_90d`, `ctr`, `avg_position`, `competition_level` – observable at the decision month and strong signals of page health.         |
| **Label**  | `trend_direction` → binary **`is_declining_label = (trend_direction == "down")`**. This proxy tells whether the page will decline in the **next month** (April 2026). |
| **Context**| `content_type`, `main_intent`, `client_id` – useful for downstream segment analysis (e.g., “how does competition affect different intents?”).               |
| **Excluded**| `provider_used`, `model_used`, `ai_sessions_90d` – these are internal ML‑system signals that are not part of the organic search‑console data and would give an unfair advantage if used. |


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check schema and field null counts


## 3. Verify it with queries (grain, counts, missing values, windows)



In [9]:
# ============================================================
# 0️⃣ FIX GRAIN — aggregate daily rows up to monthly grain
# ============================================================
march_df = (
    march_df.groupby(["content_id", "client_id", "month"], as_index=False)
    .agg(
        impressions_90d=("impressions_90d", "sum"),
        clicks_90d=("clicks_90d", "sum"),
        avg_position=("avg_position", "mean"),   # mean of daily positions -> ignores NaN days by default
    )
)
march_df["ctr"] = np.where(
    march_df["impressions_90d"] > 0,
    march_df["clicks_90d"] / march_df["impressions_90d"],
    np.nan,
)
march_df["ctr"] = march_df["ctr"].fillna(march_df["ctr"].median())

print("✅ Rebuilt march_df at monthly grain")
print(" • Shape:", march_df.shape)
print(" • Max rows per content_id:", march_df["content_id"].value_counts().max())

# ============================================================
# ✅ VERIFICATION SUITE — Grain, Counts, Missing Values, Windows
# ============================================================

TARGET_MONTH = "2026-03"
KEY_COLS = ["content_id", "month"]

results = {}

# ------------------------------------------------------------
# 1️⃣ GRAIN CHECK
# ------------------------------------------------------------
grain_check = march_df.groupby(KEY_COLS).size().reset_index(name="rows_per_key")
n_unique_pairs = grain_check.shape[0]
n_duplicates   = int(grain_check["rows_per_key"].gt(1).sum())

print("\n🔎 Grain verification")
print(f" • Distinct {tuple(KEY_COLS)} pairs : {n_unique_pairs:,}")
print(f" • Keys with duplicate rows          : {n_duplicates}")

results["grain_unique"] = (n_duplicates == 0)
if n_duplicates > 0:
    print("   ⚠️ Sample offending keys:")
    display(grain_check[grain_check["rows_per_key"] > 1].head())

# ------------------------------------------------------------
# 2️⃣ ROW COUNTS
# ------------------------------------------------------------
total_rows  = df.shape[0]
march_rows  = march_df.shape[0]

print("\n📈 Row-count summary")
print(f" • Total rows in loaded (daily) dataset : {total_rows:,}")
print(f" • Rows for target month (monthly grain): {march_rows:,}")

results["has_rows"] = (march_rows > 0)

# ------------------------------------------------------------
# 3️⃣ MISSING VALUES
# ------------------------------------------------------------
missing_counts = march_df.isna().sum()
missing_pct    = (missing_counts / max(march_rows, 1) * 100).round(2)

missing_df = (
    pd.DataFrame({"missing_vals": missing_counts, "pct_missing": missing_pct})
    .sort_values("missing_vals", ascending=False)
)

print("\n❓ Missing-value summary (target month)")
display(missing_df.head(15))

results["no_critical_missing"] = bool((missing_df.loc[KEY_COLS, "missing_vals"] == 0).all()) \
    if all(c in missing_df.index for c in KEY_COLS) else False

# ------------------------------------------------------------
# 4️⃣ WINDOW CHECK
# ------------------------------------------------------------
full_month_range  = (df["month"].min(), df["month"].max())
slice_months      = sorted(march_df["month"].unique().tolist())

print("\n🗓️ Temporal window")
print(f" • Full dataset span   : {full_month_range[0]} → {full_month_range[1]}")
print(f" • Slice month(s)      : {slice_months}")

results["window_matches_target"] = (slice_months == [TARGET_MONTH])

# ------------------------------------------------------------
# 5️⃣ NUMERIC RANGE CHECK — NaN-safe
# ------------------------------------------------------------
numeric_cols = march_df.select_dtypes(include="number").columns
range_summary = march_df[numeric_cols].agg(["min", "max"]).T

print("\n📏 Numeric column ranges")
display(range_summary)

range_flags = {}
# .dropna() before the range check — missing values are already
# tracked separately in the missing-value check above, so a NaN
# here is not a "negative value" violation.
if "ctr" in march_df.columns:
    range_flags["ctr_in_[0,1]"] = march_df["ctr"].dropna().between(0, 1).all()
if "avg_position" in march_df.columns:
    range_flags["avg_position_non_negative"] = (march_df["avg_position"].dropna() >= 0).all()
if "impressions_90d" in march_df.columns:
    range_flags["impressions_non_negative"] = (march_df["impressions_90d"].dropna() >= 0).all()
if "clicks_90d" in march_df.columns:
    range_flags["clicks_non_negative"] = (march_df["clicks_90d"].dropna() >= 0).all()

results.update(range_flags)

# ------------------------------------------------------------
# 6️⃣ FINAL SUMMARY
# ------------------------------------------------------------
print("\n" + "=" * 50)
print("✅ VERIFICATION SUMMARY")
print("=" * 50)
all_passed = True
for check, passed in results.items():
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f" {status}  — {check}")
    all_passed &= bool(passed)

print("=" * 50)
if all_passed:
    print("🎉 All checks passed — data contract satisfied.")
else:
    print("⚠️ One or more checks failed — see details above.")

assert all_passed, "Data verification failed — see summary above for which check(s) broke."

✅ Rebuilt march_df at monthly grain
 • Shape: (331437, 7)
 • Max rows per content_id: 1

🔎 Grain verification
 • Distinct ('content_id', 'month') pairs : 331,437
 • Keys with duplicate rows          : 0

📈 Row-count summary
 • Total rows in loaded (daily) dataset : 9,841,378
 • Rows for target month (monthly grain): 331,437

❓ Missing-value summary (target month)


,missing_vals,pct_missing
avg_position,154699,46.68
client_id,0,0.00
content_id,0,0.00
month,0,0.00
impressions_90d,0,0.00
clicks_90d,0,0.00
ctr,0,0.00



🗓️ Temporal window
 • Full dataset span   : 2026-03 → 2026-03
 • Slice month(s)      : ['2026-03']

📏 Numeric column ranges


,min,max
impressions_90d,0.0,617124.0
clicks_90d,0.0,5668.0
avg_position,0.0,309.0
ctr,0.0,1.0



✅ VERIFICATION SUMMARY
 ✅ PASS  — grain_unique
 ✅ PASS  — has_rows
 ✅ PASS  — no_critical_missing
 ✅ PASS  — window_matches_target
 ✅ PASS  — ctr_in_[0,1]
 ✅ PASS  — avg_position_non_negative
 ✅ PASS  — impressions_non_negative
 ✅ PASS  — clicks_non_negative
🎉 All checks passed — data contract satisfied.


## 4. Data limits

* **No true temporal granularity** – the table only carries a `month` column, not a day‑level `date`.  
  → We cannot build day‑by‑day trend lines or rolling‑window forecasts.

* **Unbalanced historical coverage** – early months (2025‑early‑2026) have far fewer rows because the ingestion pipeline was still being built.

  → Any long‑term trend must be interpreted with caution.
* **Search‑Console‑only early rows** – for the first few months only Search Console signals are present; other columns (e.g., `ai_traffic_pct`) are missing.

  → Feature completeness varies over time.
  
* **Window overlap** – because `month` aggregates all activity for a calendar month, a single `content_id` can appear in multiple months with different signal values, preventing a true “single‑snapshot” view.  
  → The slice cannot answer “what was the exact state on day X”.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check feature distributions and zero-value constraints


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.